In [1]:
### Training with manually updating W with "Backward" ###

import torch
#from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim




data = [(1.0,2.1,3.0), (2.0, 3.5, 6.0), (3.0, 3.0, 9.0), (4.0, 2.1, 12.0), (5.0, 7.2, 15.0), (6.0, 10.1, 18.0)]


class Net(nn.Module):
    def __init__(self):
        torch.manual_seed(0)
        super(Net, self).__init__()
        self.fc1 = nn.Linear(2, 1, bias=False)

    def forward(self, x):
        return self.fc1(x)

net = Net()

print(net)
print(list(net.parameters()))

#input = torch.randn(1)
#out = net(input)

#def criterion(out, label):
#    return (label - out)**2
criterion = nn.MSELoss()


optimizer = optim.SGD(net.parameters(), lr=0.01, momentum=0.5)
#optimizer = optim.Adam(net.parameters(), lr=0.005)

# All six samples are used to measure MSE after each epoch.
inputs = torch.tensor([[x1, x2] for x1, x2, y in data], dtype=torch.float32)
targets = torch.tensor([[y] for x1, x2, y in data], dtype=torch.float32)
loss_history = []
with torch.no_grad():
    initial_mse = criterion(net(inputs), targets).item()
print(f"Initial MSE: {initial_mse:.6f}")

net.train()
for epoch in range(100):
    for x1, x2, y in data:
        X = torch.tensor([x1, x2], dtype=torch.float32)
        Y = torch.tensor([y], dtype=torch.float32)
        optimizer.zero_grad()  # Clear gradients from the previous sample.
        outputs = net(X)
        loss = criterion(outputs, Y)
        loss.backward()        # Compute gradients for both weights.
        optimizer.step()       # Update weights using SGD with momentum.

    with torch.no_grad():
        epoch_mse = criterion(net(inputs), targets).item()
    loss_history.append(epoch_mse)
    if epoch == 0 or (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1:3d} - MSE: {epoch_mse:.8e}")

# Test the trained network on the supplied training samples.
net.eval()
with torch.no_grad():
    predictions = net(inputs)
    final_mse = criterion(predictions, targets).item()
    w1, w2 = net.fc1.weight[0].tolist()

print(f"\nLearned model: y = {w1:.6f} * x1 + {w2:.6f} * x2")
print(f"Final training MSE: {final_mse:.3e}")
print("\nx1     x2      target   prediction")
for (x1, x2, y), prediction in zip(data, predictions):
    print(f"{x1:4.1f}   {x2:5.1f}   {y:7.1f}   {prediction.item():10.6f}")


Net(
  (fc1): Linear(in_features=2, out_features=1, bias=False)
)
[Parameter containing:
tensor([[-0.0053,  0.3793]], requires_grad=True)]


Initial MSE: 94.615715
Epoch   1 - MSE: 3.71837120e+01
Epoch  10 - MSE: 5.88038862e-02
Epoch  20 - MSE: 2.42577426e-05
Epoch  30 - MSE: 1.29758371e-09
Epoch  40 - MSE: 9.09494702e-13
Epoch  50 - MSE: 5.84539838e-12
Epoch  60 - MSE: 5.84539838e-12
Epoch  70 - MSE: 5.76013326e-12
Epoch  80 - MSE: 5.76013326e-12
Epoch  90 - MSE: 5.76013326e-12
Epoch 100 - MSE: 5.84539838e-12

Learned model: y = 3.000000 * x1 + -0.000001 * x2
Final training MSE: 5.845e-12

x1     x2      target   prediction
 1.0     2.1       3.0     2.999999
 2.0     3.5       6.0     5.999998
 3.0     3.0       9.0     8.999999
 4.0     2.1      12.0    12.000000
 5.0     7.2      15.0    14.999996
 6.0    10.1      18.0    17.999996
